In [2]:
import influxdb_client, os, time
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd
from datetime import datetime, timedelta, timezone


INFLUXDB_TOKEN="replace_with_your_token"

token = os.environ.get("INFLUXDB_TOKEN")
org = "EST"
url = "http://localhost:8086"

client = influxdb_client.InfluxDBClient(url=url, token=INFLUXDB_TOKEN, org=org)


In [7]:
df = pd.read_csv("../../results/csv/simulation_data.csv", delimiter=',') # all the data
traj_df = pd.read_csv("../../results/csv/trajectory_coords.csv", delimiter=',') # the trajectory data

# we artificially add a timestamp to the data to convert to datetime
now = datetime(2025, 1, 1, 0, 0, 0)
df['times_telecom'] = df['times_telecom'].apply(lambda x: now + timedelta(seconds=x))

# the full df
df = pd.concat([df.reset_index(drop=True), traj_df.reset_index(drop=True)], axis=1)


In [8]:
# Write data to InfluxDB
bucket="NICE"
write_api = client.write_api(write_options=SYNCHRONOUS)
delete_api = client.delete_api()

# Delete all previous data from bucket
start = "1970-01-01T00:00:00Z"
stop =  datetime(2070, 1, 1, 0, 0, 0)
delete_api.delete(start, stop, '', bucket=bucket, org=org)



# the subsystems' times are not included because for now they are the same for all
# the .tag are used to index the data, and can be used for filtering


# Pre-build list of points
points = [
    Point("satellite_data")
        .tag("mode", int(row["modes"]))
        .tag("visible", int(row["visibility"]))
        .field("visibility", float(row["visibility"]))
        .field("data", float(row["data"]))
        .field("data_GNSS_TOF", float(row["data_GNSS_TOF"]))
        .field("data_HK", float(row["data_HK"]))
        .field("battery", float(row["battery"]))
        .field("consumption", float(row["consumption"]))
        .field("generation", float(row["generation"]))
        .field("eclipse", float(row["eclipse"]))
        .field("modes", float(row["modes"]))
        .field("altitude", float(row["altitude"]))
        .field("RAAN", float(row["RAAN"]))
        .field("AOP", float(row["AOP"]))
        .field("ECC", float(row["ECC"]))
        .field("INC", float(row["INC"]))
        .field("density", float(row["density"]))
        .field("Lat", float(row["latitude_deg"]))
        .field("Lng", float(row["longitude_deg"]))
        .field("solar_cells_efficiency", float(row["solar_cells_efficiency"]))
        .time(row["times_telecom"], write_precision=WritePrecision.NS)
    for _, row in df.iterrows()
]

# Send all points at once
write_api.write(bucket=bucket, org=org, record=points)

print("Upload complete.")

NewConnectionError: <urllib3.connection.HTTPConnection object at 0x1698478e0>: Failed to establish a new connection: [Errno 61] Connection refused